In [9]:
import os

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from utils.cfd_wrapper import OpenFoamWrapper

#### DATA

In [10]:
#### Data
sea_states = pd.read_csv('data/sea_states.txt',sep='\t')
sea_states.columns = ['hs', 'hs_l0', 'swl']

sea_states_problem = [3, 622, 936, 1115]

In [11]:
sea_states_cases = sea_states.loc[sea_states_problem]

In [12]:
sea_states_cases

,hs,hs_l0,swl
3,0.325645,0.000644,0.108267
622,0.620265,0.001709,-0.067394
936,0.422535,0.001144,-0.283049
1115,1.081972,0.005899,-0.415677


#### Inputs

In [13]:
cases_dir = 'outputs/molokai_cases_problem_refined'
templates_dir = 'inputs/templates/molokai'
outputs_dir = 'outputs/molokai_problem_refined'

In [14]:
sea_states_cases['tp'] = np.sqrt((sea_states_cases["hs"].values * 2 * np.pi) / (9.806 * sea_states_cases["hs_l0"]))
sea_states_cases['tpsoft'] = 2 * sea_states_cases['tp']
sea_states_cases['depth'] = 15 + sea_states_cases['swl']
sea_states_cases['lowfreqcutoff'] = 1 / ( 3 * sea_states_cases['tp'])
sea_states_cases['uppfreqcutoff'] = 3/  sea_states_cases['tp']

In [15]:
metamodel_parameters = sea_states_cases[['hs', 'tp', 'swl', 'tpsoft', 'depth', 'lowfreqcutoff', 'uppfreqcutoff']].to_dict(orient="list")

fixed_parameters = {'points_per_wavelenght':200,
                    'domain_lenght':2000,
                    'points_per_waveheight':15,
                    'domain_height':5,
                    #'alpha_inlet_patch_vals':[],
                    'total_run_time': 3600,
                    #'boundary_file':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/templates/openfoam/constant/polyMesh/boundary',
                    #'block_mesh_dict':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/templates/openfoam/constant/polyMesh/blockMeshDict',
                    'preprocess_script':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/scripts_openfoam/preprocess_case.sh',
                    'createmesh_script':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/scripts_openfoam/createmesh_case.sh',
                    'postprocess_script':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/scripts_openfoam/postprocess_case.sh'}

openfoam_wrapper = OpenFoamWrapper(
    templates_dir = templates_dir,
    metamodel_parameters = metamodel_parameters,
    fixed_parameters = fixed_parameters,
    output_dir = cases_dir,
)

2026-03-26 03:58:24,362 - OpenFoamWrapper - WARNING - Parameter hs is not in the default_parameters
2026-03-26 03:58:24,363 - OpenFoamWrapper - WARNING - Parameter tp is not in the default_parameters
2026-03-26 03:58:24,363 - OpenFoamWrapper - WARNING - Parameter swl is not in the default_parameters
2026-03-26 03:58:24,363 - OpenFoamWrapper - WARNING - Parameter tpsoft is not in the default_parameters
2026-03-26 03:58:24,363 - OpenFoamWrapper - WARNING - Parameter depth is not in the default_parameters
2026-03-26 03:58:24,364 - OpenFoamWrapper - WARNING - Parameter lowfreqcutoff is not in the default_parameters
2026-03-26 03:58:24,364 - OpenFoamWrapper - WARNING - Parameter uppfreqcutoff is not in the default_parameters


In [16]:
#openfoam_wrapper.build_cases()

In [18]:
openfoam_wrapper.save_model(model_path=os.path.join(outputs_dir,'openfoam_model.pkl'), exclude_attributes=['_env'])